# Final cleaned Kaggle notebook — from-scratch TTS
This notebook is the **clean version** you should run in a **new Kaggle notebook / fresh session**.

What it fixes:
- no phonemizer dependency
- no pandas dependency
- metadata preprocessing happens in a subprocess
- data is streamed from JSONL using file offsets
- corrected Griffin-Lim dtype handling
- corrected `Audio(filename=...)`
- stronger training settings for alignment

This is still **from scratch**:
- random model initialization
- no pretrained checkpoints
- trained only on your attached dataset


In [ ]:
# =========================
# 1) Minimal install
# =========================
!pip -q install unidecode soundfile
print("Dependencies installed.")


In [ ]:
# =========================
# 2) Imports + seed
# =========================
import os
import re
import sys
import json
import math
import time
import random
import warnings
import subprocess
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
from unidecode import unidecode

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 3) Config
# =========================
SAMPLE_RATE = 22050
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 256
N_MELS = 80
FMIN = 0
FMAX = 8000

MIN_TEXT_LEN = 1
MAX_TEXT_LEN = 220

VAL_SIZE = 300
BATCH_SIZE = 8
NUM_WORKERS = 0
PIN_MEMORY = True

EMBED_DIM = 256
ENC_HIDDEN = 256
DEC_HIDDEN = 512
ATTN_DIM = 128
LOCATION_FILTERS = 32
LOCATION_KERNEL = 31
PRENET_DIM = 256
POSTNET_CHANNELS = 512
POSTNET_KERNEL = 5
REDUCTION_FACTOR = 2

EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 5
USE_AMP = torch.cuda.is_available()

TEACHER_FORCING_START = 1.0
TEACHER_FORCING_END = 0.90

GUIDED_ATTN_MAX_LAMBDA = 0.15
GUIDED_ATTN_MIN_LAMBDA = 0.02
GUIDED_ATTN_G = 0.2
STOP_POS_WEIGHT = 8.0

DEFAULT_STOP_THRESHOLD = 0.60
DEFAULT_MIN_DECODER_STEPS = 30
DEFAULT_MAX_DECODER_STEPS = 800

WORK_DIR = Path("/kaggle/working/final_tts_clean")
CKPT_DIR = WORK_DIR / "checkpoints"
SAMPLE_DIR = WORK_DIR / "samples"

for p in [WORK_DIR, CKPT_DIR, SAMPLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Work dir:", WORK_DIR)


In [ ]:
# =========================
# 4) Find LJSpeech dataset
# =========================
def find_ljspeech_root(search_root="/kaggle/input"):
    candidates = []
    for root, dirs, files in os.walk(search_root):
        if "metadata.csv" in files and "wavs" in dirs:
            candidates.append(root)
    return candidates

candidates = find_ljspeech_root()
if not candidates:
    raise FileNotFoundError("Could not find LJSpeech dataset with metadata.csv and wavs/")

print("Found candidates:")
for c in candidates:
    print(" -", c)

LJSPEECH_PATH = Path(candidates[0])
CSV_PATH = LJSPEECH_PATH / "metadata.csv"
WAVS_DIR = LJSPEECH_PATH / "wavs"
print("Using:", LJSPEECH_PATH)


In [ ]:
# =========================
# 5) Preprocess metadata in a subprocess -> JSONL
# =========================
PROCESSED_META = str(WORK_DIR / "ljs_meta_processed.jsonl")

script = rf"""
import os
import json
import re
from unidecode import unidecode

CSV_PATH = r"{str(CSV_PATH)}"
WAVS_DIR = r"{str(WAVS_DIR)}"
OUT_PATH = r"{PROCESSED_META}"

MIN_TEXT_LEN = {MIN_TEXT_LEN}
MAX_TEXT_LEN = {MAX_TEXT_LEN}

_whitespace_re = re.compile(r"\s+")
_allowed_text = re.compile(r"[^a-zA-Z0-9 ,.!?;:'\\"()\\-\\[\\]]+")

def clean_text(text: str) -> str:
    text = str(text)
    text = unidecode(text)
    text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    text = _allowed_text.sub(" ", text)
    text = _whitespace_re.sub(" ", text).strip()
    return text

def text_to_token_string(text: str) -> str:
    text = clean_text(text)
    return " ".join(list(text))

count = 0
with open(CSV_PATH, "r", encoding="utf-8", errors="ignore") as f, open(OUT_PATH, "w", encoding="utf-8") as g:
    for line in f:
        parts = line.rstrip("\\n").split("|")
        if len(parts) < 2:
            continue

        utt_id = parts[0]
        raw_text = parts[1] if len(parts) > 1 else ""
        norm_text = parts[2] if len(parts) > 2 else raw_text

        wav_path = os.path.join(WAVS_DIR, f"{{utt_id}}.wav")
        if not os.path.exists(wav_path):
            continue

        text = clean_text(norm_text if norm_text else raw_text)
        if not (MIN_TEXT_LEN <= len(text) <= MAX_TEXT_LEN):
            continue

        tok_str = text_to_token_string(text)
        if len(tok_str) == 0:
            continue

        rec = {{
            "id": utt_id,
            "text": text,
            "wav_path": wav_path,
            "tokens": tok_str
        }}
        g.write(json.dumps(rec, ensure_ascii=False) + "\\n")
        count += 1

print("Processed rows:", count)
print("Saved to:", OUT_PATH)
"""

subprocess.run([sys.executable, "-c", script], check=True)


In [ ]:
# =========================
# 6) Build file offsets only
# =========================
offsets = []
with open(PROCESSED_META, "rb") as f:
    while True:
        pos = f.tell()
        line = f.readline()
        if not line:
            break
        offsets.append(pos)

print("Rows in JSONL:", len(offsets))

random.shuffle(offsets)
val_size = min(VAL_SIZE, max(100, len(offsets) // 20))
val_offsets = offsets[:val_size]
train_offsets = offsets[val_size:]

print("Train rows:", len(train_offsets))
print("Val rows:", len(val_offsets))


In [ ]:
# =========================
# 7) Build vocab by streaming JSONL once
# =========================
special_tokens = ["<pad>", "<sos>", "<eos>"]
token_set = set()

with open(PROCESSED_META, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        for t in item["tokens"].split():
            token_set.add(t)

vocab = special_tokens + sorted(token_set)
stoi = {s: i for i, s in enumerate(vocab)}
itos = {i: s for s, i in stoi.items()}

PAD_ID = stoi["<pad>"]
SOS_ID = stoi["<sos>"]
EOS_ID = stoi["<eos>"]

print("Vocab size:", len(vocab))
print("First tokens:", vocab[:20])


In [ ]:
# =========================
# 8) Audio helpers
# =========================
def load_wav(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    wav = np.clip(wav, -1.0, 1.0)
    return wav

def wav_to_log_mel(wav):
    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=1.0
    )
    mel = np.log(np.maximum(mel, 1e-5)).astype(np.float32)
    return mel.T

def log_mel_to_audio_griffinlim(mel_log, n_iter=100):
    mel_log = mel_log.astype(np.float32)
    mel = np.exp(mel_log.T)
    audio = librosa.feature.inverse.mel_to_audio(
        mel,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        power=1.0,
        n_iter=n_iter
    )
    return np.clip(audio, -1.0, 1.0).astype(np.float32)


In [ ]:
# =========================
# 9) Dataset and collate
# =========================
class JSONLTTSDataset(Dataset):
    def __init__(self, jsonl_path, offsets):
        self.jsonl_path = jsonl_path
        self.offsets = offsets

    def __len__(self):
        return len(self.offsets)

    def token_string_to_ids(self, token_str):
        toks = token_str.split()
        ids = [SOS_ID] + [stoi[t] for t in toks if t in stoi] + [EOS_ID]
        return np.array(ids, dtype=np.int64)

    def __getitem__(self, idx):
        offset = self.offsets[idx]
        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            f.seek(offset)
            item = json.loads(f.readline())

        ids = self.token_string_to_ids(item["tokens"])
        wav = load_wav(item["wav_path"])
        mel = wav_to_log_mel(wav)

        return {
            "ids": ids,
            "mel": mel,
            "text": item["text"],
            "tokens": item["tokens"],
        }

def collate_batch(batch):
    batch = sorted(batch, key=lambda x: len(x["ids"]), reverse=True)

    input_lens = [len(x["ids"]) for x in batch]
    mel_lens = [x["mel"].shape[0] for x in batch]

    max_input_len = max(input_lens)
    max_mel_len = max(mel_lens)
    if max_mel_len % REDUCTION_FACTOR != 0:
        max_mel_len += REDUCTION_FACTOR - (max_mel_len % REDUCTION_FACTOR)

    B = len(batch)
    ids = torch.full((B, max_input_len), PAD_ID, dtype=torch.long)
    mels = torch.zeros((B, max_mel_len, N_MELS), dtype=torch.float32)
    stop_targets = torch.zeros((B, max_mel_len // REDUCTION_FACTOR), dtype=torch.float32)

    texts, tokens = [], []
    for i, item in enumerate(batch):
        cur_ids = torch.tensor(item["ids"], dtype=torch.long)
        cur_mel = torch.tensor(item["mel"], dtype=torch.float32)

        ids[i, :cur_ids.size(0)] = cur_ids
        mels[i, :cur_mel.size(0)] = cur_mel

        reduced_len = math.ceil(cur_mel.size(0) / REDUCTION_FACTOR)
        stop_targets[i, reduced_len - 1:] = 1.0

        texts.append(item["text"])
        tokens.append(item["tokens"])

    return (
        ids,
        torch.tensor(input_lens, dtype=torch.long),
        mels,
        torch.tensor(mel_lens, dtype=torch.long),
        stop_targets,
        texts,
        tokens,
    )

train_loader = DataLoader(
    JSONLTTSDataset(PROCESSED_META, train_offsets),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    JSONLTTSDataset(PROCESSED_META, val_offsets),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

print("Dataloaders ready.")


In [ ]:
# =========================
# 10) Model
# =========================
class Prenet(nn.Module):
    def __init__(self, in_dim, sizes=(256, 256), dropout=0.5):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(in_dim if i == 0 else sizes[i-1], sizes[i]) for i in range(len(sizes))]
        )
        self.dropout = dropout

    def forward(self, x):
        for linear in self.layers:
            x = F.relu(linear(x))
            x = F.dropout(x, p=self.dropout, training=True)
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, embed_dim, kernel_size=5, padding=2),
                nn.BatchNorm1d(embed_dim),
                nn.ReLU(),
                nn.Dropout(0.5)
            ) for _ in range(3)
        ])
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x, lengths):
        x = self.embedding(x).transpose(1, 2)
        for conv in self.convs:
            x = conv(x)
        x = x.transpose(1, 2)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        packed_out, _ = self.bilstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        return out

class LocationLayer(nn.Module):
    def __init__(self, attn_n_filters, attn_kernel_size, attn_dim):
        super().__init__()
        padding = (attn_kernel_size - 1) // 2
        self.location_conv = nn.Conv1d(2, attn_n_filters, kernel_size=attn_kernel_size, padding=padding, bias=False)
        self.location_dense = nn.Linear(attn_n_filters, attn_dim, bias=False)

    def forward(self, attn_cat):
        return self.location_dense(self.location_conv(attn_cat).transpose(1, 2))

class LocationSensitiveAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.query_layer = nn.Linear(dec_dim, attn_dim, bias=False)
        self.memory_layer = nn.Linear(enc_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=True)
        self.location_layer = LocationLayer(LOCATION_FILTERS, LOCATION_KERNEL, attn_dim)
        self.score_mask_value = -float("inf")

    def get_alignment_energies(self, query, processed_memory, attn_cat):
        processed_query = self.query_layer(query).unsqueeze(1)
        processed_location = self.location_layer(attn_cat)
        energies = self.v(torch.tanh(processed_query + processed_location + processed_memory)).squeeze(-1)
        return energies

    def forward(self, query, memory, processed_memory, attention_weights_cat, mask):
        alignment = self.get_alignment_energies(query, processed_memory, attention_weights_cat)
        if mask is not None:
            alignment.data.masked_fill_(mask, self.score_mask_value)
        attn_weights = F.softmax(alignment, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), memory).squeeze(1)
        return context, attn_weights

class Decoder(nn.Module):
    def __init__(self, enc_dim, mel_dim, prenet_dim, dec_hidden, attn_dim):
        super().__init__()
        self.prenet = Prenet(mel_dim, sizes=(prenet_dim, prenet_dim), dropout=0.5)
        self.attn_rnn = nn.LSTMCell(prenet_dim + enc_dim, dec_hidden)
        self.attention = LocationSensitiveAttention(enc_dim, dec_hidden, attn_dim)
        self.decoder_rnn = nn.LSTMCell(dec_hidden + enc_dim, dec_hidden)
        proj_in = dec_hidden + enc_dim
        self.mel_proj = nn.Linear(proj_in, mel_dim * REDUCTION_FACTOR)
        self.stop_proj = nn.Linear(proj_in, 1)

    def initialize_states(self, memory, mask):
        B, T_enc, enc_dim = memory.size()
        self.memory = memory
        self.processed_memory = self.attention.memory_layer(memory)
        self.mask = mask

        self.attn_hidden = memory.new_zeros(B, DEC_HIDDEN)
        self.attn_cell = memory.new_zeros(B, DEC_HIDDEN)
        self.dec_hidden = memory.new_zeros(B, DEC_HIDDEN)
        self.dec_cell = memory.new_zeros(B, DEC_HIDDEN)
        self.attn_weights = memory.new_zeros(B, T_enc)
        self.attn_weights_cum = memory.new_zeros(B, T_enc)
        self.context = memory.new_zeros(B, enc_dim)

    def decode_step(self, prev_mel):
        prenet_out = self.prenet(prev_mel)
        attn_input = torch.cat([prenet_out, self.context], dim=-1)

        self.attn_hidden, self.attn_cell = self.attn_rnn(attn_input, (self.attn_hidden, self.attn_cell))
        self.attn_hidden = F.dropout(self.attn_hidden, 0.1, self.training)

        attn_cat = torch.stack([self.attn_weights, self.attn_weights_cum], dim=1)
        self.context, self.attn_weights = self.attention(
            self.attn_hidden, self.memory, self.processed_memory, attn_cat, self.mask
        )
        self.attn_weights_cum = self.attn_weights_cum + self.attn_weights

        dec_input = torch.cat([self.attn_hidden, self.context], dim=-1)
        self.dec_hidden, self.dec_cell = self.decoder_rnn(dec_input, (self.dec_hidden, self.dec_cell))
        self.dec_hidden = F.dropout(self.dec_hidden, 0.1, self.training)

        proj_input = torch.cat([self.dec_hidden, self.context], dim=-1)
        mel_out = self.mel_proj(proj_input)
        stop_logit = self.stop_proj(proj_input).squeeze(-1)
        return mel_out, stop_logit, self.attn_weights

class Postnet(nn.Module):
    def __init__(self, mel_dim=80, channels=512, kernel_size=5, num_layers=5):
        super().__init__()
        layers = []
        in_ch = mel_dim
        for i in range(num_layers):
            out_ch = channels if i < num_layers - 1 else mel_dim
            layers.append(
                nn.Sequential(
                    nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=(kernel_size - 1) // 2),
                    nn.BatchNorm1d(out_ch),
                )
            )
            in_ch = out_ch
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        x = x.transpose(1, 2)
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = torch.tanh(x)
            x = F.dropout(x, 0.5, self.training)
        return x.transpose(1, 2)

class FromScratchTTS(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = Encoder(vocab_size, EMBED_DIM, ENC_HIDDEN)
        self.decoder = Decoder(ENC_HIDDEN * 2, N_MELS, PRENET_DIM, DEC_HIDDEN, ATTN_DIM)
        self.postnet = Postnet(N_MELS, POSTNET_CHANNELS, POSTNET_KERNEL, 5)

    def forward(
        self,
        inputs,
        input_lens,
        target_mels=None,
        teacher_forcing_ratio=1.0,
        max_decoder_steps=DEFAULT_MAX_DECODER_STEPS,
        stop_threshold=DEFAULT_STOP_THRESHOLD,
        min_decoder_steps=DEFAULT_MIN_DECODER_STEPS
    ):
        B = inputs.size(0)
        memory = self.encoder(inputs, input_lens)
        max_enc_len = memory.size(1)
        mask = torch.arange(max_enc_len, device=inputs.device).unsqueeze(0) >= input_lens.unsqueeze(1)

        self.decoder.initialize_states(memory, mask)

        steps = math.ceil(target_mels.size(1) / REDUCTION_FACTOR) if target_mels is not None else max_decoder_steps

        prev_mel = torch.zeros(B, N_MELS, device=inputs.device)
        mel_outputs, stop_outputs, attn_outputs = [], [], []

        for step in range(steps):
            mel_out, stop_logit, attn = self.decoder.decode_step(prev_mel)
            mel_frame = mel_out.view(B, REDUCTION_FACTOR, N_MELS)

            mel_outputs.append(mel_frame)
            stop_outputs.append(stop_logit)
            attn_outputs.append(attn)

            if target_mels is not None and random.random() < teacher_forcing_ratio:
                idx = min((step + 1) * REDUCTION_FACTOR - 1, target_mels.size(1) - 1)
                prev_mel = target_mels[:, idx, :]
            else:
                prev_mel = mel_frame[:, -1, :]

            if target_mels is None and step >= min_decoder_steps:
                if torch.sigmoid(stop_logit).mean().item() > stop_threshold:
                    break

        mel_outputs = torch.cat(mel_outputs, dim=1)
        stop_outputs = torch.stack(stop_outputs, dim=1)
        attn_outputs = torch.stack(attn_outputs, dim=1)
        mel_post = mel_outputs + self.postnet(mel_outputs)
        return mel_outputs, mel_post, stop_outputs, attn_outputs


In [ ]:
# =========================
# 11) Losses and schedules
# =========================
def make_nonpad_mask(lengths, max_len):
    ids = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return ids < lengths.unsqueeze(1)

def masked_l1_loss(pred, target, lengths):
    max_t = min(pred.size(1), target.size(1))
    pred = pred[:, :max_t, :]
    target = target[:, :max_t, :]
    lengths = torch.clamp(lengths, max=max_t)

    mask = make_nonpad_mask(lengths, max_t).unsqueeze(-1).float()
    loss = torch.abs(pred - target) * mask
    return loss.sum() / (mask.sum() * pred.size(-1) + 1e-8)

def masked_stop_bce(stop_logits, stop_targets, mel_lens):
    reduced_lens = torch.ceil(mel_lens.float() / REDUCTION_FACTOR).long()
    T = stop_logits.size(1)
    reduced_lens = torch.clamp(reduced_lens, max=T)

    mask = make_nonpad_mask(reduced_lens, T).float()
    loss = F.binary_cross_entropy_with_logits(
        stop_logits,
        stop_targets[:, :T],
        reduction="none",
        pos_weight=torch.tensor(STOP_POS_WEIGHT, device=stop_logits.device),
    )
    loss = loss * mask
    return loss.sum() / (mask.sum() + 1e-8)

def guided_attention_loss(attn, input_lens, mel_lens, g=GUIDED_ATTN_G):
    B, _, _ = attn.shape
    total = 0.0
    count = 0

    for b in range(B):
        T_dec = min(attn.size(1), math.ceil(mel_lens[b].item() / REDUCTION_FACTOR))
        T_enc = min(attn.size(2), input_lens[b].item())
        if T_dec <= 1 or T_enc <= 1:
            continue

        t = torch.arange(T_dec, device=attn.device).unsqueeze(1).float() / T_dec
        n = torch.arange(T_enc, device=attn.device).unsqueeze(0).float() / T_enc
        W = 1.0 - torch.exp(-((n - t) ** 2) / (2 * g * g))
        A = attn[b, :T_dec, :T_enc]

        total += torch.mean(A * W)
        count += 1

    if count == 0:
        return torch.tensor(0.0, device=attn.device)
    return total / count

def current_teacher_forcing(epoch, total_epochs):
    if total_epochs <= 1:
        return TEACHER_FORCING_END
    alpha = (epoch - 1) / (total_epochs - 1)
    return TEACHER_FORCING_START + alpha * (TEACHER_FORCING_END - TEACHER_FORCING_START)

def current_guided_lambda(epoch, total_epochs):
    if total_epochs <= 1:
        return GUIDED_ATTN_MIN_LAMBDA
    alpha = (epoch - 1) / (total_epochs - 1)
    return GUIDED_ATTN_MAX_LAMBDA + alpha * (GUIDED_ATTN_MIN_LAMBDA - GUIDED_ATTN_MAX_LAMBDA)


In [ ]:
# =========================
# 12) Build model, optimizer, checkpoints
# =========================
model = FromScratchTTS(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1, min_lr=5e-5
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"

def save_checkpoint(path, epoch, best_val, history):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_val": best_val,
        "history": history,
        "vocab": vocab,
    }, path)

history = {
    "train_total": [],
    "val_total": [],
}

start_epoch = 1
best_val = float("inf")
epochs_without_improvement = 0


In [ ]:
# =========================
# 13) Inference helpers
# =========================
def token_ids_from_text(text):
    text = str(text)
    text = unidecode(text)
    tokens = " ".join(list(text))
    toks = tokens.split()
    ids = [SOS_ID] + [stoi[t] for t in toks if t in stoi] + [EOS_ID]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0), tokens

@torch.no_grad()
def synthesize_mel(model, text):
    model.eval()
    ids, tok_str = token_ids_from_text(text)
    ids = ids.to(device)
    lengths = torch.tensor([ids.size(1)], dtype=torch.long, device=device)

    with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
        _, mel_post, _, attn = model(
            ids,
            lengths,
            target_mels=None,
            teacher_forcing_ratio=0.0,
            max_decoder_steps=DEFAULT_MAX_DECODER_STEPS,
            stop_threshold=DEFAULT_STOP_THRESHOLD,
            min_decoder_steps=DEFAULT_MIN_DECODER_STEPS,
        )

    return {
        "mel": mel_post.squeeze(0).detach().cpu().float().numpy(),
        "tokens": tok_str,
        "attn": attn.squeeze(0).detach().cpu().float().numpy(),
    }


In [ ]:
# =========================
# 14) Training loop
# =========================
def validate(epoch):
    model.eval()
    total_loss_sum = 0.0
    total_items = 0

    for batch in val_loader:
        ids, input_lens, mels, mel_lens, stop_targets, _, _ = batch
        ids = ids.to(device)
        input_lens = input_lens.to(device)
        mels = mels.to(device)
        mel_lens = mel_lens.to(device)
        stop_targets = stop_targets.to(device)

        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            mel_pred, mel_post, stop_logits, attn = model(
                ids, input_lens, target_mels=mels, teacher_forcing_ratio=1.0
            )
            mel_loss = masked_l1_loss(mel_pred, mels, mel_lens) + masked_l1_loss(mel_post, mels, mel_lens)
            stop_loss = masked_stop_bce(stop_logits, stop_targets, mel_lens)
            attn_loss = guided_attention_loss(attn, input_lens, mel_lens)
            total_loss = mel_loss + stop_loss + current_guided_lambda(epoch, EPOCHS) * attn_loss

        bs = ids.size(0)
        total_loss_sum += total_loss.item() * bs
        total_items += bs

    return total_loss_sum / max(total_items, 1)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    tf_ratio = current_teacher_forcing(epoch, EPOCHS)
    attn_lambda = current_guided_lambda(epoch, EPOCHS)

    train_loss_sum = 0.0
    total_items = 0

    for batch in train_loader:
        ids, input_lens, mels, mel_lens, stop_targets, _, _ = batch
        ids = ids.to(device)
        input_lens = input_lens.to(device)
        mels = mels.to(device)
        mel_lens = mel_lens.to(device)
        stop_targets = stop_targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            mel_pred, mel_post, stop_logits, attn = model(
                ids, input_lens, target_mels=mels, teacher_forcing_ratio=tf_ratio
            )
            mel_loss = masked_l1_loss(mel_pred, mels, mel_lens) + masked_l1_loss(mel_post, mels, mel_lens)
            stop_loss = masked_stop_bce(stop_logits, stop_targets, mel_lens)
            attn_loss = guided_attention_loss(attn, input_lens, mel_lens)
            total_loss = mel_loss + stop_loss + attn_lambda * attn_loss

        if not torch.isfinite(total_loss):
            continue

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        bs = ids.size(0)
        train_loss_sum += total_loss.item() * bs
        total_items += bs

    train_loss = train_loss_sum / max(total_items, 1)
    val_loss = validate(epoch)
    scheduler.step(val_loss)

    history["train_total"].append(train_loss)
    history["val_total"].append(val_loss)
    save_checkpoint(LATEST_CKPT, epoch, best_val, history)

    if val_loss < best_val:
        best_val = val_loss
        epochs_without_improvement = 0
        save_checkpoint(BEST_CKPT, epoch, best_val, history)
        tag = "Saved best."
    else:
        epochs_without_improvement += 1
        tag = f"No improvement ({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})"

    print(
        f"Epoch {epoch:02d} | Train {train_loss:.4f} | Val {val_loss:.4f} | "
        f"TF {tf_ratio:.3f} | Attn λ {attn_lambda:.3f} | {tag}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping.")
        break


In [ ]:
# =========================
# 15) Final inference
# =========================
if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("Loaded best checkpoint.")
else:
    print("Best checkpoint not found. Using current weights.")

TEST_TEXT = "Hello, this is my final text to speech model for my graduation project."
final_wav = WORK_DIR / "final_demo.wav"

result = synthesize_mel(model, TEST_TEXT)
mel = result["mel"].astype(np.float32)
tokens = result["tokens"]

audio = log_mel_to_audio_griffinlim(mel, n_iter=100)
sf.write(final_wav, audio, SAMPLE_RATE)

print("Saved:", final_wav)
print("Tokens:", tokens[:200])
print("Audio duration (sec):", len(audio) / SAMPLE_RATE)


In [ ]:
# =========================
# 16) Play saved audio
# =========================
from IPython.display import Audio, display
display(Audio(filename=str(final_wav)))
